# 아이템 기반 협업 필터링 (Item-based Collaborative Filtering)

**개념**

* **아이템 간 유사도**를 기반으로 추천을 제공하는 방식
* 사용자의 **과거 행동 데이터**를 활용

**추천 절차**

1. **아이템 간 유사도 계산**

   * 코사인 유사도, 피어슨 상관 계수 등 사용
2. **사용자 선호 아이템 확인**
3. **예측 평점 계산**

   * 유사 아이템의 평점을 가중 평균
4. **추천 생성**

   * 예측 평점이 높은 아이템 추천

**장점**

* 사용자 수가 많아도 **계산 효율성**이 높음
* 아이템 특성 없이도 **추천 가능**

**단점**

* **개인화 한계** (사용자 취향 반영 어려움)
* **Cold Start 문제** 발생 가능 (데이터 부족 시)


## 데이터 준비 
https://grouplens.org/datasets/movielens/

`recommended for education and development` 섹션 하위 [ml-latest-small.zip](https://files.grouplens.org/datasets/movielens/ml-latest-small.zip) 다운로드

In [1]:
import pandas as pd
import numpy as np

In [4]:
movies_df = pd.read_csv('./data/ml-latest-small/movies.csv')
ratings_df = pd.read_csv('./data/ml-latest-small/ratings.csv')

print(movies_df.shape, ratings_df.shape)

movies_df.head()

(9742, 3) (100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [6]:
# 평점 기준 0.5 ~ 5점
ratings_df['rating'].describe()

count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64

In [7]:
# 영화 정보, 사용자 정보 병합
movies_rating_df = pd.merge(ratings_df, movies_df, on='movieId', how='inner')

print(movies_rating_df.shape)

movies_rating_df.head()

(100836, 6)


,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


## 사용자 평점기반 아이템(영화) 유사도 계산 

1. 사용자-아이템 행렬을 아이템-사용자 행렬로 변경한다.
2. 아이템간 코사인 유사도로 계산해서 아이템-아이템 행렬을 만든다.

In [8]:
# 사용자 X 아이템(영화) 행렬
user_movie_df = movies_rating_df.pivot_table('rating', index='userId', columns='title', fill_value=0)   # NaN을 0으로 채우기
print(user_movie_df)

user_movie_df.head()

title   '71 (2014)  'Hellboy': The Seeds of Creation (2004)  \
userId                                                        
1              0.0                                      0.0   
2              0.0                                      0.0   
3              0.0                                      0.0   
4              0.0                                      0.0   
5              0.0                                      0.0   
...            ...                                      ...   
606            0.0                                      0.0   
607            0.0                                      0.0   
608            0.0                                      0.0   
609            0.0                                      0.0   
610            4.0                                      0.0   

title   'Round Midnight (1986)  'Salem's Lot (2004)  \
userId                                                
1                          0.0                  0.0   
2              

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# 아이템(영화) X 사용자 행렬
movie_user_df = user_movie_df.T
print(movie_user_df.shape)

movie_user_df.head()

(9719, 610)


userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
'Hellboy': The Seeds of Creation (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Round Midnight (1986),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Salem's Lot (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Til There Was You (1997),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

movie_sim = cosine_similarity(movie_user_df, movie_user_df)
print(movie_user_df)

# 영화 제목을 index, column에 모두 지정
movie_sim_df = pd.DataFrame(movie_sim, index=user_movie_df.columns, columns=user_movie_df.columns)
movie_sim_df.head()

userId                                     1    2    3    4    5    6    7    \
title                                                                          
'71 (2014)                                 0.0  0.0  0.0  0.0  0.0  0.0  0.0   
'Hellboy': The Seeds of Creation (2004)    0.0  0.0  0.0  0.0  0.0  0.0  0.0   
'Round Midnight (1986)                     0.0  0.0  0.0  0.0  0.0  0.0  0.0   
'Salem's Lot (2004)                        0.0  0.0  0.0  0.0  0.0  0.0  0.0   
'Til There Was You (1997)                  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
...                                        ...  ...  ...  ...  ...  ...  ...   
eXistenZ (1999)                            0.0  0.0  0.0  0.0  0.0  0.0  0.0   
xXx (2002)                                 0.0  0.0  0.0  0.0  0.0  0.0  0.0   
xXx: State of the Union (2005)             0.0  0.0  0.0  0.0  0.0  0.0  0.0   
¡Three Amigos! (1986)                      4.0  0.0  0.0  0.0  0.0  0.0  0.0   
À nous la liberté (Freedom for Us) (1931

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.141653,0.0,...,0.0,0.342055,0.543305,0.707107,0.0,0.0,0.139431,0.327327,0.0,0.0
'Hellboy': The Seeds of Creation (2004),0.0,1.000000,0.707107,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Round Midnight (1986),0.0,0.707107,1.000000,0.000000,0.000000,0.0,0.176777,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Salem's Lot (2004),0.0,0.000000,0.000000,1.000000,0.857493,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
'Til There Was You (1997),0.0,0.000000,0.000000,0.857493,1.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0


## 가중 평점 예측
1. 사용자별 아이템(영화) 평점이 있다. `user_rating_df`
2. 평점 기반의 아이템(영화) 유사도가 있다. `movie_sim_df`
3. 모든 사용자의 모든 아이템(영화)에 대한 가중평점을 예측한다. 
4. 사용자별로 예측한 가중평점이 높은 순으로 영화를 추천한다. 
    - 사용자가 안본 영화(평점이 없는 영화)만 필터링해서 추천한다. 

**Weighted Rating Sum**

사용자 $ u $의 아이템 $ i $에 대한 평점 예측은 사용자 $ u $가 아이템 $ i $와 유사한 다른 아이템들($ N $개의 다른 아이템)의 합으로 계산하며, 아이템들 간의 유사도를 반영한 합으로 계산.

$
\hat{R}_{u,i} = \frac{\sum_{N} (S_{i,N} \times R_{u,N})}{\sum_{N} (|S_{i,N}|)}
$


- $\hat{R}_{u,i}$: 사용자 $ u $가 아이템 $ i $에 대해 가질 것으로 예측되는 평점
- $S_{i,N}$: 아이템 $ i $와 유사한 다른 아이템들의 유사도
- $R_{u,N}$: 사용자 $u$의 유사한 아이템들의 평점

**사용자 $ u $의 유사한 아이템 들의 평점**

| Item | j | k | **i** | m | n |
|------|---|---|---|---|---|
| Rating | 5 | 4 | **1** | 3 | 2 |

**유사도**

|   | (i,j) | (i,k) | (i,i) | (i,m) | (i,n) |
|---|-------|-------|-------|-------|-------|
| **R_{u,i}** | 0.2 | 0.1 | **0.4** | 0.1 | 0.2 |

**계산:** 위 두개의 행렬에 대한 내적을 구한다.

$
5 \times 0.2 + 4 \times 0.1 + 1 \times 0.4 + 3 \times 0.1 + 2 \times 0.2 = 2.5
$
<br>
<br>
$
\hat{R}_{u,i} = \frac{(5 \times 0.2) + (4 \times 0.1) + (1 \times 0.4) + (3 \times 0.1) + (2 \times 0.2)}{0.2 + 0.1 + 0.4 + 0.1 + 0.2}
= \frac{1 + 0.4 + 0.4 + 0.3 + 0.4}{1} = \frac{2.5}{1} = 2.5
$

결과적으로, 사용자 $ u $가 아이템 $ i $에 대해 가질 것으로 예측되는 평점은 **2.5**입니다.


### 전체 가중평점 예측하기1 - (610 * 9719)
전체 영화를 대상으로 유사도를 구하므로 정확도가 좋지 않다. 

In [14]:
def predict_ratings_by_all_movie(user_movie_df, movie_sim_df):
    '''
    :param user_movie_df : R 사용자 평점
    :param movie_sim_df : S 유사도
    '''
    # 사용자-영화 평점 행렬과 영화-영화 유사도 행렬을 곱하여 사용자-영화 예측 평점 행렬을 구할 수 있다.
    # 유사도 행렬의 각 행의 절대값의 합으로 나누어 정규화 한다.
    return (user_movie_df @ movie_sim_df) / np.abs(movie_sim_df).sum(axis=1)

rating_pred_df = predict_ratings_by_all_movie(user_movie_df, movie_sim_df)

# 1번 사용자의 가중 평점 예측
rating_pred_df.head(1)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.070345,0.577855,0.321696,0.227055,0.206958,0.194615,0.249883,0.102542,0.157084,0.178197,...,0.113608,0.181738,0.133962,0.128574,0.006179,0.21207,0.192921,0.136024,0.292955,0.720347


In [17]:
from sklearn.metrics import mean_squared_error
# 실제 평점과 예측 평점의 오차 비교
def movie_rating_mse(actual, pred):
    # 실제로 평가한 영화 (0이 아닌 값)만 추출하여 MSE 계산
    non_zero_idx = actual.nonzero() # 0이 아닌 값의 인덱스
    actual = actual[non_zero_idx]
    pred = pred[non_zero_idx]
    return mean_squared_error(actual, pred)

movie_rating_mse(user_movie_df.values, rating_pred_df.values)   # ndarray 변환해서 전달

9.895354759094706

### 특정 사용자의 특정 영화 평점 예측
- 영화별 평점 유사도를 기반해서 가중평점 예측하기 

In [18]:
# 35번 영화 조회
movie_user_df.iloc[35]
movie_sim_df.iloc[35]

title
'71 (2014)                                   0.0
'Hellboy': The Seeds of Creation (2004)      0.0
'Round Midnight (1986)                       0.0
'Salem's Lot (2004)                          0.0
'Til There Was You (1997)                    0.0
                                            ... 
eXistenZ (1999)                              0.0
xXx (2002)                                   0.0
xXx: State of the Union (2005)               0.0
¡Three Amigos! (1986)                        0.0
À nous la liberté (Freedom for Us) (1931)    0.0
Name: 12 Angry Men (1997), Length: 9719, dtype: float64

In [19]:
# 176번 사용자의 35번 영화에 대한 예측
# 실제 사용자의 평점은 5점
user_movie_df.iloc[176, 35]

5.0

In [21]:
# 35번 영화와 유사한 영화 인덱스 20개 조회
topn = 20
topn_sim_idx = movie_sim_df.iloc[35].argsort()[::-1].to_numpy()
topn_sim_idx = topn_sim_idx[topn_sim_idx != 35] # 자기자신 제외
topn_sim_idx = topn_sim_idx[:topn]
topn_sim_idx

array([5855, 9491, 3522, 4129, 6814, 4412, 8656, 1098, 4426, 5658, 9169,
       3523, 8416, 8495, 7788, 8784, 9609, 2357,  502,  740], dtype=int64)

In [22]:
movie_sim_df.iloc[35, :].iloc[topn_sim_idx]

title
Mr. Skeffington (1944)                                     1.000000
Winchester '73 (1950)                                      1.000000
Gold Diggers of 1933 (1933)                                1.000000
Hunchback of Notre Dame, The (1923)                        1.000000
Princess and the Pirate, The (1944)                        1.000000
Intolerance: Love's Struggle Throughout the Ages (1916)    1.000000
Thief of Bagdad, The (1924)                                1.000000
Birth of a Nation, The (1915)                              1.000000
Invisible Man Returns, The (1940)                          1.000000
Mildred Pierce (2011)                                      1.000000
Very Potter Sequel, A (2010)                               1.000000
Gold Diggers of 1935 (1935)                                1.000000
The Blue Lagoon (1949)                                     1.000000
The Great Train Robbery (1903)                             0.970143
Snake Pit, The (1948)                     

In [ ]:
# 176의 사용자의 실제 영화 평점


In [ ]:
# 사용자 1명, 영화 1개에 대한 예측 평점 계산
def predict_ratings_user_personalized(user_idx, movie_idx, topn_sim_idx):
    
    # 영화별 평점 유사도 조회
    topn_sim = movie_sim[movie_idx, : ][topn_sim_idx]
    
    # 평점 유사도 topn건에 대한 사용자 평점
    topn_rating = user_movie_df.values[user_idx, :][topn_sim_idx]
    
    # 사용자가 실제로 본 영화만 선택(생략)
    
    
    # 예측 평점 계산
    return (topn_sim @ topn_rating) / np.abs(topn_sim).sum()

topn = 20
user_idx = 176
movie_idx = 35
topn_sim_idx = movie_sim_df.iloc[movie_idx].argsort()[:-(topn+2):-1]
topn_sim_idx = topn_sim_idx[topn_sim_idx != movie_idx]

predict_ratings_user_personalized(user_idx, movie_idx, topn_sim_idx)

3.7104590164872704

### 전체 가중평점 예측하기2 - (610 * 9719) - 영화별 사용자별 평점유사도 상위 topn건만 적용

In [36]:
# 모든 사용자가 모든 영화에 대해 가질 예측 평점을 한 번에 계산하는 함수
def predict_ratings(topn=20):
    ratings = user_movie_df.values
    sim = movie_sim_df.values
    
    num_users, num_movies = ratings.shape
    pred = np.zeros((num_users, num_movies))    # 예측 결과로 사용할 배열
    
    for movie_idx in range(num_movies):
        # 현재 영화와 유사도 높은 영화 인덱스 정렬
        topn_sim_idx = np.argsort(sim[movie_idx][::-1])
        
        # 자기 자신 제외 후 topn 선택
        topn_sim_idx = topn_sim_idx[topn_sim_idx != movie_idx][:topn]
        
        # 현재 영화와 topn 유사 영화들의 유사도
        topn_sim = sim[movie_idx, topn_sim_idx]
        
        # 모든 사용자의 topn 유사 영화들의 평점
        topn_ratings = ratings[:, topn_sim_idx]
        
        # 사용자가 실제로 본 영화만 반영
        non_zero_mask = topn_ratings != 0
        
        # 분자 : 유사도 * 평점의 합
        numer = (topn_ratings * topn_sim * non_zero_mask).sum(axis=1)
        
        # 분모 = 실제로 본 유사 영화들에 해당하는 유사도 절댓값 합
        denom = (np.abs(topn_sim) * non_zero_mask).sum(axis=1)
        
        # 0으로 나누기 방지
        pred[:, movie_idx] = np.divide(
            numer,
            denom,
            out = np.zeros(num_users),
            where=denom != 0
        )
        
    return pred

ratings_pred = predict_ratings()
# print(ratings_pred.shape)

In [37]:
# 특정 사용자의 예측 평점 조회
ratings_pred[176,35]

4.0

In [38]:
# 예측 평점 평가
movie_rating_mse(user_movie_df.values, ratings_pred)

7.5510410057820785

In [39]:
# 예측 평점 df
rating_pred_df = pd.DataFrame(ratings_pred, index=user_movie_df.index, columns=user_movie_df.columns)
rating_pred_df.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,4.0,3.0,...,0.0,0.0,0.0,3.646577,0.0,5.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


### 사용자별 안본 영화 조회

In [40]:
def get_unseen_movies(user_idx):
    # user_idx 번째 사용자의 전체 영화 평점
    user_ratings_df = user_movie_df.iloc[user_idx]
    # 평점이 0인 영화만 반환
    return user_ratings_df[user_ratings_df == 0]

### 사용자별 영화추천 

In [41]:
def recommend_movies(user_idx, topn=20):
    # user_idx번째 사용자가 안 본 영화 제목 목록
    unseen = get_unseen_movies(user_idx).index
    print(unseen)
    # 해당 사용자의 예측 평점 중 안 본 영화만 골라 높은 순으로 정렬
    temp = rating_pred_df.iloc[user_idx][unseen].sort_values(ascending=False)[:topn]
    # 추천 된 영화들에 대한 실제 평점 확인
    user_rating_df = user_movie_df.iloc[user_idx][temp.index]
    
    return pd.DataFrame({
        'title' : temp.index,
        'pred_rating' : temp.values,
        'user_rating' : user_rating_df.values
    })
    
# 100번째 행의 사용자의 영화 추천 결과 조회
recommend_movies(100)

Index([''71 (2014)', ''Hellboy': The Seeds of Creation (2004)',
       ''Round Midnight (1986)', ''Salem's Lot (2004)',
       ''Til There Was You (1997)', ''Tis the Season for Love (2015)',
       ''burbs, The (1989)', ''night Mother (1986)',
       '(500) Days of Summer (2009)', '*batteries not included (1987)',
       ...
       'Zulu (1964)', 'Zulu (2013)', '[REC] (2007)', '[REC]² (2009)',
       '[REC]³ 3 Génesis (2012)',
       'anohana: The Flower We Saw That Day - The Movie (2013)', 'xXx (2002)',
       'xXx: State of the Union (2005)', '¡Three Amigos! (1986)',
       'À nous la liberté (Freedom for Us) (1931)'],
      dtype='str', name='title', length=9658)


,title,pred_rating,user_rating
0,Little Man Tate (1991),5.0,0.0
1,Super Mario Bros. (1993),5.0,0.0
2,Manhattan Murder Mystery (1993),5.0,0.0
3,Lara Croft: Tomb Raider (2001),5.0,0.0
4,Assassins (1995),5.0,0.0
5,One Crazy Summer (1986),5.0,0.0
6,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",5.0,0.0
7,Battlestar Galactica (2003),5.0,0.0
8,To Catch a Thief (1955),5.0,0.0
9,Dirty Rotten Scoundrels (1988),5.0,0.0


In [42]:
def recommend_movies(user_idx, topn=20):
    # user_idx번째 사용자가 안 본 영화 제목 목록
    unseen = get_unseen_movies(user_idx).index   # index>영화제목
    # 해당 사용자의 예측 평점 중 안본 영화만 골라 높은 순으로 정렬
    temp = rating_pred_df.iloc[user_idx][unseen].sort_values(ascending=False)[:topn]
    # 추천 된 영화들에 대한 실제 평점 확인
    user_rating_df = user_movie_df.iloc[user_idx][temp.index]

    return pd.DataFrame({
        'title' : temp.index,
        'pred_rating' : temp.values,
        'user_rating' : user_rating_df.values
    })
# 100번째 행의 사용자의 영화 추천 결과 조회
recommend_movies(100)

,title,pred_rating,user_rating
0,Little Man Tate (1991),5.0,0.0
1,Super Mario Bros. (1993),5.0,0.0
2,Manhattan Murder Mystery (1993),5.0,0.0
3,Lara Croft: Tomb Raider (2001),5.0,0.0
4,Assassins (1995),5.0,0.0
5,One Crazy Summer (1986),5.0,0.0
6,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",5.0,0.0
7,Battlestar Galactica (2003),5.0,0.0
8,To Catch a Thief (1955),5.0,0.0
9,Dirty Rotten Scoundrels (1988),5.0,0.0
